# Dataset Download and Quick Inspection

1. Download raw benchmark datasets (Yoochoose and Diginetica) from Kaggle.
2. Store untouched artifacts in `data/raw/`.
3. Preview table information (fields, shape, sample rows).
4. Write download manifest with source, date, file names, and sizes.

Requirements:
- Install python packages from `requirements.txt`.
- Kaggle access configured for your environment (required by `kagglehub`).

In [ ]:
import json
import shutil
from pathlib import Path

import kagglehub
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
def find_file(root_dir, filename):
    exact_matches = list(root_dir.rglob(filename))
    if exact_matches:
        return exact_matches[0]

    filename_lower = filename.lower()
    for path in root_dir.rglob("*"):
        if path.is_file() and path.name.lower() == filename_lower:
            return path

    return None


DATASETS = [
    {
        "name": "diginetica",
        "kaggle_dataset": "profalbusdumbledore/diginetica-dataset",
        "tables": [
            {
                "table_name": "item_views",
                "filename": "train-item-views.csv",
                "has_header": True,
                "fields": [
                    "session_id",
                    "item_id",
                    "timeframe",
                    "eventdate",
                ],
            }
        ],
    },
    {
        "name": "yoochoose",
        "kaggle_dataset": "phhasian0710/yoochoose",
        "tables": [
            {
                "table_name": "clicks",
                "filename": "yoochoose-clicks.dat",
                "has_header": False,
                "fields": ["session_id", "timestamp", "item_id", "category"],
            }
        ],
    },
]

manifest = []
PREPARED_TABLES = []

for dataset in DATASETS:
    print(f"Dataset: {dataset['name']}")

    kaggle_cache_path = Path(kagglehub.dataset_download(dataset["kaggle_dataset"]))

    raw_dataset_dir = RAW_DIR / dataset["name"]
    raw_dataset_dir.mkdir(parents=True, exist_ok=True)

    dataset_file_names = []
    dataset_file_sizes = {}

    for table in dataset["tables"]:
        source_path = find_file(kaggle_cache_path, table["filename"])
        copied_path = None

        if source_path is None:
            print(f"Missing file in Kaggle dataset: {table['filename']}")
        else:
            copied_path = raw_dataset_dir / table["filename"]
            shutil.copy2(source_path, copied_path)
            rel_path = str(copied_path.relative_to(RAW_DIR))
            dataset_file_names.append(rel_path)
            dataset_file_sizes[rel_path] = copied_path.stat().st_size

        PREPARED_TABLES.append(
            {
                "dataset": dataset["name"],
                "table_name": table["table_name"],
                "fields": table["fields"],
                "has_header": table["has_header"],
                "path": copied_path,
            }
        )

    manifest.append(
        {
            "dataset": dataset["name"],
            "kaggle_dataset": dataset["kaggle_dataset"],
            "raw_file_names": dataset_file_names,
        }
    )

manifest_path = RAW_DIR / "download_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

Dataset: diginetica
Dataset: yoochoose


328

In [27]:
def preview_table(table_info):
    if table_info["path"] is None:
        print("\n" + "-" * 80)
        print(
            f"{table_info['dataset']} / {table_info['table_name']}: file not found after extraction."
        )
        return

    file_path = Path(table_info["path"])
    fields = table_info["fields"]
    has_header = table_info["has_header"]

    if has_header:
        df_head = pd.read_csv(file_path, nrows=5)
        actual_fields = list(df_head.columns)
    else:
        df_head = pd.read_csv(file_path, header=None, names=fields, nrows=5)
        actual_fields = fields

    print("\n" + "-" * 80)
    print(f"Dataset: {table_info['dataset']}")
    print(f"Size: {file_path.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"Table: {table_info['table_name']}")
    print(f"Fields ({df_head.shape[1]}): {actual_fields}")
    print("Sample rows:")
    print(df_head)


for table in PREPARED_TABLES:
    preview_table(table)


--------------------------------------------------------------------------------
Dataset: diginetica
Size: 40.69 MB
Table: item_views
Fields (1): ['sessionId;userId;itemId;timeframe;eventdate']
Sample rows:
  sessionId;userId;itemId;timeframe;eventdate
0                1;NA;81766;526309;2016-05-09
1               1;NA;31331;1031018;2016-05-09
2                1;NA;32118;243569;2016-05-09
3                  1;NA;9654;75848;2016-05-09
4               1;NA;32627;1112408;2016-05-09

--------------------------------------------------------------------------------
Dataset: yoochoose
Size: 1417.92 MB
Table: clicks
Fields (4): ['session_id', 'timestamp', 'item_id', 'category']
Sample rows:
   session_id                 timestamp    item_id  category
0           1  2014-04-07T10:51:09.277Z  214536502         0
1           1  2014-04-07T10:54:09.868Z  214536500         0
2           1  2014-04-07T10:54:46.998Z  214536506         0
3           1  2014-04-07T10:57:00.306Z  214577561         0
4  